In [1]:
import os
import argparse
import random
import time
import warnings
warnings.filterwarnings("ignore")
 
import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms.functional import rotate as tvrotate
from torchvision import transforms
from tqdm.auto import tqdm

def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
 
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def get_args():
    p = argparse.ArgumentParser(description="Bonus Task – Exp 2: Orbit Averaging")
    p.add_argument("--task1_ckpt", type=str, default="/kaggle/input/models/vitthal2945/e2e-vaemodel/pytorch/default/1/best_model.pt")
    p.add_argument("--task3_ckpt", type=str, default="/kaggle/input/models/vitthal2945/best-mode-task3/pytorch/default/1/best_model_task3.pt")
    p.add_argument("--clf_ckpt",   type=str, default="/kaggle/input/models/vitthal2945/classifier-exp1/pytorch/default/1/classifiers_exp1.pt",
                   help="Classifier checkpoint from exp1 (noaug key)")
    p.add_argument("--data_dir",   type=str, default="/kaggle/input/datasets/vitthal2945/e2e-hidden-symmetries-dataset")
    p.add_argument("--out_dir",    type=str, default="./bonus_outputs/exp2")
    p.add_argument("--latent_dim", type=int, default=16)
    p.add_argument("--n_classes",  type=int, default=2,
                   help="Number of digit classes (set 2 if Task 3 was trained on digits 1&2)")
    p.add_argument("--eps",        type=float, default=0.05,
                   help="Integration step size ε for orbit sampling")
    p.add_argument("--orbit_k",    type=int, default=12,
                   help="Number of orbit steps K to integrate per generator")
    return p.parse_args([])

In [3]:
class ResBlock(nn.Module):
    def __init__(self, ch, g=8):
        super().__init__()
        g = min(g, ch)
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch),
        )
        self.act = nn.SiLU()
    def forward(self, x): return self.act(x + self.net(x))
 
class Encoder(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.stem  = nn.Sequential(nn.Conv2d(1, 32, 4, 2, 1, bias=False), nn.GroupNorm(8, 32), nn.SiLU())
        self.l1    = nn.Sequential(ResBlock(32),  nn.Conv2d(32,  64, 4, 2, 1, bias=False), nn.GroupNorm(8,  64), nn.SiLU())
        self.l2    = nn.Sequential(ResBlock(64),  nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.GroupNorm(8, 128), nn.SiLU())
        self.l3    = ResBlock(128)
        self.fc    = nn.Sequential(nn.Flatten(), nn.Linear(128*4*4, 512), nn.SiLU())
        self.fc_mu = nn.Linear(512, ld); self.fc_lv = nn.Linear(512, ld)
    def forward(self, x):
        h = self.l3(self.l2(self.l1(self.stem(x)))); h = self.fc(h)
        return self.fc_mu(h), self.fc_lv(h)
 
class Decoder(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ld, 512), nn.SiLU(), nn.Linear(512, 128*4*4), nn.SiLU())
        self.u1 = nn.Sequential(ResBlock(128), nn.ConvTranspose2d(128, 64, 3, 2, 1, output_padding=1), nn.GroupNorm(8,  64), nn.SiLU())
        self.u2 = nn.Sequential(ResBlock(64),  nn.ConvTranspose2d(64,  32, 4, 2, 1),                  nn.GroupNorm(8,  32), nn.SiLU())
        self.u3 = nn.Sequential(ResBlock(32),  nn.ConvTranspose2d(32,   1, 4, 2, 3),                  nn.Sigmoid())
    def forward(self, z): return self.u3(self.u2(self.u1(self.fc(z).view(-1, 128, 4, 4))))
 
class Task1VAE(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.encoder = Encoder(ld); self.decoder = Decoder(ld)
    def encode(self, x): mu, _ = self.encoder(x); return mu
    def decode(self, z): return self.decoder(z)
 
 
class SpectralRefinementGenerator(nn.Module):
    """Exact replica of Task-3 SpectralRefinementGenerator (v4)."""
    def __init__(self, ld: int, global_dir: torch.Tensor, hidden: int = 256):
        super().__init__()
        self.register_buffer("v", F.normalize(global_dir.float(), dim=0))
        self.log_scale = nn.Parameter(torch.tensor(-2.3))
        self.delta = nn.Sequential(
            nn.Linear(ld, hidden),     nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, ld),
        )
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        scale = torch.exp(self.log_scale)
        delta = scale * self.delta(z)
        return self.v.unsqueeze(0).expand(len(z), -1) + delta
 
 
class MLPClassifier(nn.Module):
    def __init__(self, ld: int, nc: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ld, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, nc),
        )
    def forward(self, z): return self.net(z)

In [4]:
def load_vae(path, ld):
    vae = Task1VAE(ld).to(device)
    if os.path.exists(path):
        ck = torch.load(path, map_location=device, weights_only=False)
        vae.load_state_dict(ck.get("model", ck))
        print(f"  ✓ VAE loaded (epoch {ck.get('epoch', '?')})")
    else:
        print(f"  ⚠  VAE ckpt not found: {path}")
    vae.eval()
    for p in vae.parameters(): p.requires_grad = False
    return vae
 
def load_generators(path, ld):
    """
    Load Task-3 generators from best_model.pt.
    Returns (generators list, SEMs list, global_dirs tensor).
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Task-3 checkpoint not found: {path}")
    ck = torch.load(path, map_location=device, weights_only=False)
 
    global_dirs  = ck["global_dirs"]        # (NG, ld) tensor or list
    eigenvalues  = ck["eigenvalues"]        # full eigenvalue spectrum
    gen_states   = ck["generators"]         # list of state dicts
 
    NG = len(gen_states)
    if not isinstance(global_dirs, torch.Tensor):
        global_dirs = torch.tensor(global_dirs)
    SEMs = eigenvalues[:NG].tolist() if hasattr(eigenvalues, "__iter__") else [1.0] * NG
 
    generators = []
    for i, sd in enumerate(gen_states):
        G = SpectralRefinementGenerator(ld, global_dirs[i]).to(device)
        G.load_state_dict(sd)
        G.eval()
        for p in G.parameters(): p.requires_grad = False
        generators.append(G)
 
    print(f"  ✓ Task-3 generators loaded: {NG} generators")
    for i, (G, sem) in enumerate(zip(generators, SEMs)):
        print(f"      G_{i+1}  SEM = {sem:.4f}")
    return generators, SEMs, global_dirs
 
def load_clf(path, ld, nc, key="noaug"):
    clf = MLPClassifier(ld, nc).to(device)
    if os.path.exists(path):
        ck  = torch.load(path, map_location=device, weights_only=False)
        sd  = ck.get(key, ck)
        clf.load_state_dict(sd)
        print(f"  ✓ Classifier ('{key}') loaded from {path}")
    else:
        print(f"  ⚠  Classifier ckpt not found: {path} – using random weights")
    clf.eval()
    for p in clf.parameters(): p.requires_grad = False
    return clf

In [5]:
def load_h5(path):
    print(f"  loading {os.path.basename(path)} ...", end=" ", flush=True)
    t0 = time.time()
    with h5py.File(path, "r") as f:
        imgs   = torch.from_numpy(f["images"][:].astype("float32")).unsqueeze(1).clamp(0., 1.)
        labels = torch.from_numpy(f["labels"][:].astype("int64"))
        angles = torch.from_numpy(f["angles"][:].astype("int64"))
    print(f"done ({time.time()-t0:.1f}s)")
    return imgs, labels, angles

In [6]:
@torch.no_grad()
def rk2_step(z: torch.Tensor, G: SpectralRefinementGenerator, eps: float) -> torch.Tensor:
    """Second-order Runge-Kutta (midpoint method) orbit integration step."""
    g0    = F.normalize(G(z), dim=1)
    z_mid = z + (eps / 2) * g0
    g_mid = F.normalize(G(z_mid), dim=1)
    return z + eps * g_mid
 
@torch.no_grad()
def sample_orbit(
    z0: torch.Tensor,
    G:  SpectralRefinementGenerator,
    K:  int,
    eps: float
) -> torch.Tensor:
    """
    Integrate K steps of the generator G starting from z0.
    Returns tensor of shape (B, K+1, ld) — includes z0 as step 0.
    """
    B, ld = z0.shape
    orbit = [z0]
    z_cur = z0.clone()
    for _ in range(K):
        z_cur = rk2_step(z_cur, G, eps)
        orbit.append(z_cur)
    return torch.stack(orbit, dim=1)   # (B, K+1, ld)

In [7]:
@torch.no_grad()
def orbit_predict_single_gen(
    clf:   MLPClassifier,
    z0:    torch.Tensor,
    G:     SpectralRefinementGenerator,
    K:     int,
    eps:   float
) -> torch.Tensor:
    """
    Uniform orbit average over one generator.
    Returns softmax predictions (B, C).
    """
    orbit = sample_orbit(z0, G, K, eps)  # (B, K+1, ld)
    B, Kp1, ld = orbit.shape
    # flatten batch×orbit, classify, unflatten
    flat_z    = orbit.view(B * Kp1, ld).to(device)
    flat_prob = F.softmax(clf(flat_z), dim=1)
    probs     = flat_prob.view(B, Kp1, -1)   # (B, K+1, C)
    return probs.mean(dim=1)                  # (B, C)
 
 
@torch.no_grad()
def swop_predict(
    clf:        MLPClassifier,
    z0:         torch.Tensor,
    generators: list,
    SEMs:       list,
    K:          int,
    eps:        float
) -> torch.Tensor:
    """
    SEM-Weighted Orbit Pooling (SWOP) — novel contribution.
 
    For each generator G_α (with SEM weight w_α):
      1. Sample K orbit points via RK2
      2. Compute average softmax over those K points → p_α(z)
    Final prediction: f_inv(z) = Σ_α w_α · p_α(z)
 
    Principle: generators with higher SEM eigenvalue are more reliable
    global symmetries → their orbital averages carry more weight.
    """
    sem_arr = np.array(SEMs, dtype=np.float32)
    weights = sem_arr / sem_arr.sum()   # normalise to sum=1
 
    pooled = None
    for G, w in zip(generators, weights):
        p_alpha = orbit_predict_single_gen(clf, z0, G, K, eps)  # (B, C)
        pooled  = p_alpha * w if pooled is None else pooled + p_alpha * w
    return pooled   # (B, C) — already a valid probability distribution
 
@torch.no_grad()
def uniform_multiGen_predict(
    clf:        MLPClassifier,
    z0:         torch.Tensor,
    generators: list,
    K:          int,
    eps:        float
) -> torch.Tensor:
    """Uniform pooling over all generators (ablation: ignores SEM weights)."""
    w = 1.0 / len(generators)
    pooled = None
    for G in generators:
        p = orbit_predict_single_gen(clf, z0, G, K, eps)
        pooled = p * w if pooled is None else pooled + p * w
    return pooled

In [8]:
@torch.no_grad()
def evaluate_strategy(
    strategy_fn,
    vae:    Task1VAE,
    imgs_0: torch.Tensor,
    lbls_0: torch.Tensor,
    angles: list,
    dig2cls: dict,
    batch:  int = 128,
    desc:   str = ""
) -> dict:
    """
    Evaluate strategy_fn across all rotations.
    strategy_fn: callable(z0) → class probabilities (B, C)
    Returns {angle: accuracy}.
    """
    results = {}
    for angle in tqdm(angles, desc=f"  eval {desc}", leave=False):
        correct = total = 0
        for i in range(0, len(imgs_0), batch):
            chunk = imgs_0[i:i+batch]
            if angle != 0:
                chunk = tvrotate(chunk, float(angle),
                                 interpolation=transforms.InterpolationMode.BILINEAR,
                                 fill=[0.])
            z0   = vae.encode(chunk.to(device))
            prob = strategy_fn(z0)                       # (B, C)
            pred = prob.argmax(dim=1).cpu()
            # map original labels to contiguous classes
            lbl_chunk = torch.tensor([dig2cls.get(l.item(), l.item())
                                      for l in lbls_0[i:i+batch]])
            correct += (pred == lbl_chunk).sum().item()
            total   += len(pred)
        results[angle] = correct / total
    return results

In [9]:
@torch.no_grad()
def generator_confidence_score(
    clf:  MLPClassifier,
    vae:  Task1VAE,
    imgs: torch.Tensor,
    G:    SpectralRefinementGenerator,
    K:    int,
    eps:  float,
    batch: int = 128
) -> float:
    """
    GCS(G) = 1 − mean_z TV(p_0(z), p_orbit_avg(z))
    where TV = total variation distance = 0.5 * ||p_0 - p_avg||_1
 
    Measures how much the orbit average changes the prediction relative to
    the base prediction — lower change = better symmetry.
    GCS ∈ [0, 1]; higher = more invariant generator.
    """
    tvs = []
    for i in range(0, min(len(imgs), 512), batch):
        chunk = imgs[i:i+batch].to(device)
        z0    = vae.encode(chunk)
        p0    = F.softmax(clf(z0), dim=1)
        p_avg = orbit_predict_single_gen(clf, z0, G, K, eps)
        tv    = 0.5 * (p0 - p_avg).abs().sum(dim=1)   # (B,)
        tvs.extend(tv.cpu().numpy().tolist())
    return float(1.0 - np.mean(tvs))

In [10]:
@torch.no_grad()
def orbit_depth_ablation(
    clf:        MLPClassifier,
    vae:        Task1VAE,
    generators: list,
    SEMs:       list,
    imgs_0:     torch.Tensor,
    lbls_0:     torch.Tensor,
    angles:     list,
    dig2cls:    dict,
    eps:        float,
    K_list:     list = [6, 12, 24]
) -> dict:
    """Returns {K: {angle: acc}} for each K in K_list."""
    results = {}
    for K in K_list:
        print(f"\n    Orbit depth K={K} …")
        fn = lambda z0, K=K: swop_predict(clf, z0, generators, SEMs, K, eps)
        results[K] = evaluate_strategy(fn, vae, imgs_0, lbls_0, angles, dig2cls,
                                       desc=f"SWOP K={K}")
    return results

In [11]:
DARK, PANEL = "#0d0d0d", "#1a1a2e"
CMAP = ["#e94560", "#0f9b8e", "#f5a623", "#9b59b6", "#2ecc71"]
 
def _ax(ax, title="", xl="", yl=""):
    ax.set_facecolor(PANEL); ax.tick_params(colors="white")
    for s in ax.spines.values(): s.set_edgecolor("#444")
    if title: ax.set_title(title, color="white", fontsize=9)
    if xl:    ax.set_xlabel(xl, color="white", fontsize=8)
    if yl:    ax.set_ylabel(yl, color="white", fontsize=8)
 
def plot_results(
    angles: list,
    acc_baseline: dict,
    acc_uniform:  dict,
    acc_swop:     dict,
    acc_depth:    dict,
    gcs:          list,
    SEMs:         list,
    out_path:     str
):
    fig = plt.figure(figsize=(22, 14), facecolor=DARK)
    fig.suptitle("Exp 2 — SEM-Weighted Orbit Averaging (SWOP)\n"
                 "Novel: Symmetry Equivariance Measure weights orbit contributions",
                 color="white", fontsize=12, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.32)
 
    x = np.array(angles)
 
    # Panel 1: main accuracy comparison
    ax1 = fig.add_subplot(gs[0, :2])
    _ax(ax1, "Per-Angle Accuracy: Baseline vs Uniform Orbit vs SWOP",
        "Rotation Angle θ (°)", "Accuracy")
    for key, acc, col, ls in [
        ("MLP-NoAug (baseline)", acc_baseline, "#555", "--"),
        ("Uniform orbit avg",    acc_uniform,  CMAP[1], "-"),
        ("SWOP (ours)",          acc_swop,     CMAP[0], "-"),
    ]:
        y = np.array([acc[a] for a in angles])
        ax1.plot(x, y, ls, color=col, lw=2.2, label=f"{key}  [mean={y.mean():.4f}]")
    ax1.set_xticks(x); ax1.set_xticklabels([f"{a}°" for a in angles], color="white", fontsize=8)
    ax1.legend(facecolor=PANEL, labelcolor="white", fontsize=9)
    ax1.set_ylim(0, 1.05)
 
    # Panel 2: SEM weights + GCS
    ax2 = fig.add_subplot(gs[0, 2])
    _ax(ax2, "Generator Quality:\nSEM (weight) vs GCS (orbit confidence)", "", "")
    NG = len(SEMs)
    sem_norm = np.array(SEMs) / np.array(SEMs).sum()
    x_gen = np.arange(NG)
    ax2b  = ax2.twinx()
    b1 = ax2.bar(x_gen - 0.2, sem_norm, width=0.35, color=CMAP[0], alpha=0.85, label="SEM weight")
    b2 = ax2b.bar(x_gen + 0.2, gcs, width=0.35, color=CMAP[1], alpha=0.85, label="GCS (↑ better)")
    ax2.set_xticks(x_gen); ax2.set_xticklabels([f"G_{i+1}" for i in range(NG)], color="white")
    ax2.set_ylabel("SEM weight", color=CMAP[0], fontsize=8)
    ax2b.set_ylabel("GCS", color=CMAP[1], fontsize=8)
    ax2b.tick_params(colors="white")
    lines = [b1[0], b2[0]]
    ax2.legend(lines, [l.get_label() for l in lines], facecolor=PANEL, labelcolor="white", fontsize=8)
    ax2.set_facecolor(PANEL); ax2.tick_params(colors="white")
 
    # Panel 3: orbit depth ablation
    ax3 = fig.add_subplot(gs[1, :2])
    _ax(ax3, "Orbit Depth Ablation: SWOP Accuracy vs K (integration steps)",
        "Rotation Angle θ (°)", "Accuracy")
    K_list = sorted(acc_depth.keys())
    for K, col in zip(K_list, CMAP):
        y = np.array([acc_depth[K][a] for a in angles])
        ax3.plot(x, y, "o-", color=col, lw=2, label=f"K={K}  [mean={y.mean():.4f}]")
    ax3.set_xticks(x); ax3.set_xticklabels([f"{a}°" for a in angles], color="white", fontsize=8)
    ax3.legend(facecolor=PANEL, labelcolor="white", fontsize=9)
    ax3.set_ylim(0, 1.05)
 
    # Panel 4: mean accuracy bar chart
    ax4 = fig.add_subplot(gs[1, 2])
    _ax(ax4, "Mean Accuracy Summary", "", "Mean Accuracy (all θ)")
    methods = ["NoAug", "Uniform", "SWOP"] + [f"SWOP K={K}" for K in K_list]
    accs_all = [
        np.mean(list(acc_baseline.values())),
        np.mean(list(acc_uniform.values())),
        np.mean(list(acc_swop.values())),
    ] + [np.mean(list(acc_depth[K].values())) for K in K_list]
    cols = ["#555", CMAP[1], CMAP[0]] + [CMAP[i+2 % len(CMAP)] for i in range(len(K_list))]
    bars = ax4.bar(range(len(methods)), accs_all, color=cols, alpha=0.85, edgecolor=DARK)
    ax4.set_xticks(range(len(methods)))
    ax4.set_xticklabels(methods, color="white", fontsize=7, rotation=30, ha="right")
    for bar, v in zip(bars, accs_all):
        ax4.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                 f"{v:.4f}", ha="center", color="white", fontsize=8)
    ax4.set_ylim(0, 1.1)
 
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")
 
 
def plot_orbit_images(
    vae:   Task1VAE,
    imgs_0: torch.Tensor,
    generators: list,
    SEMs:  list,
    eps:   float,
    K:     int = 8,
    n_samples: int = 3,
    out_path: str = ""
):
    """Visualise decoded orbit images for each generator across n_samples."""
    NG = len(generators)
    rng = np.random.default_rng(0)
    idx = rng.choice(len(imgs_0), n_samples, replace=False)
 
    fig, axes = plt.subplots(n_samples * NG, K + 1, figsize=((K+1)*1.4, n_samples*NG*1.6), facecolor=DARK)
    if n_samples * NG == 1: axes = np.array([[axes]])
    if axes.ndim == 1: axes = axes.reshape(1, -1)
    fig.suptitle(f"SWOP orbit images  (ε={eps}, K={K})\n"
                 "Each block: one sample × one generator; RK2 integration",
                 color="white", fontsize=10, fontweight="bold")
 
    cmap_g = [CMAP[i % len(CMAP)] for i in range(NG)]
    for s_i, s_idx in enumerate(idx):
        base_img = imgs_0[s_idx:s_idx+1]
        for gi, G in enumerate(generators):
            row = s_i * NG + gi
            z = vae.encode(base_img.to(device))
            for col in range(K + 1):
                ax = axes[row, col]
                img_np = vae.decode(z).clamp(0., 1.).cpu().squeeze().numpy()
                ax.imshow(img_np, cmap="plasma", vmin=0, vmax=1)
                ax.axis("off")
                if col == 0:
                    ax.set_ylabel(f"G_{gi+1}\nSEM={SEMs[gi]:.3f}",
                                  color=cmap_g[gi], fontsize=7, rotation=0, labelpad=45)
                if s_i == 0 and gi == 0:
                    ax.set_title(f"t={col}", color="white", fontsize=7)
                if col < K:
                    z = rk2_step(z, G, eps)
 
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")

In [12]:
def main():
    args = get_args()
    os.makedirs(args.out_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"  Bonus Task · Exp 2: SEM-Weighted Orbit Averaging (SWOP)")
    print(f"  device = {device}")
    print(f"  output → {args.out_dir}")
    print(f"{'='*60}\n")
 
    # ── Load models ───────────────────────────────────────────────────────────
    vae        = load_vae(args.task1_ckpt, args.latent_dim)
    generators, SEMs, _ = load_generators(args.task3_ckpt, args.latent_dim)
    NG = len(generators)
 
    # Determine n_classes from checkpoint metadata or arg
    # Task 3 may have been trained on 2 classes (digits 1&2) but we classify 10
    # clf from exp1 was trained on n_classes digits — use that
    n_classes = args.n_classes
    clf = load_clf(args.clf_ckpt, args.latent_dim, n_classes, key="noaug")
 
    # ── Load data ─────────────────────────────────────────────────────────────
    _, _, _ = None, None, None  # suppress unused warning
    test_imgs, test_lbls, test_angs = load_h5(
        os.path.join(args.data_dir, "rotated_mnist_test.h5"))
    ANGLES  = sorted(test_angs.unique().tolist())
    DIGITS  = sorted(test_lbls.unique().tolist())
    dig2cls = {d: i for i, d in enumerate(DIGITS)}
 
    # Use 0° test images as source
    mask_0  = test_angs == 0
    imgs_0  = test_imgs[mask_0]
    lbls_0  = test_lbls[mask_0]
    print(f"  Test (0° only): {len(imgs_0)} samples  |  angles: {ANGLES}")
 
    # ── Baseline: no-aug MLP on each angle ────────────────────────────────────
    print("\n[1] Evaluating baseline (MLP-NoAug, no orbit) …")
    @torch.no_grad()
    def baseline_fn(z0):
        return F.softmax(clf(z0), dim=1)
    acc_baseline = evaluate_strategy(baseline_fn, vae, imgs_0, lbls_0,
                                     ANGLES, dig2cls, desc="baseline")
 
    # ── Uniform multi-generator orbit avg ────────────────────────────────────
    print("\n[2] Evaluating uniform orbit averaging (all generators, equal weight) …")
    def uniform_fn(z0):
        return uniform_multiGen_predict(clf, z0, generators, args.orbit_k, args.eps)
    acc_uniform = evaluate_strategy(uniform_fn, vae, imgs_0, lbls_0,
                                    ANGLES, dig2cls, desc="uniform")
 
    # ── SWOP: SEM-weighted orbit averaging ───────────────────────────────────
    print("\n[3] Evaluating SWOP (SEM-weighted orbit pooling) …")
    def swop_fn(z0):
        return swop_predict(clf, z0, generators, SEMs, args.orbit_k, args.eps)
    acc_swop = evaluate_strategy(swop_fn, vae, imgs_0, lbls_0,
                                 ANGLES, dig2cls, desc="SWOP")
 
    # ── Generator Confidence Score ────────────────────────────────────────────
    print("\n[4] Computing Generator Confidence Scores (GCS) …")
    gcs_scores = []
    for i, G in enumerate(generators):
        gcs = generator_confidence_score(clf, vae, imgs_0, G, args.orbit_k, args.eps)
        gcs_scores.append(gcs)
        print(f"    G_{i+1}  SEM={SEMs[i]:.4f}  GCS={gcs:.4f}")
 
    # ── Orbit depth ablation ──────────────────────────────────────────────────
    print("\n[5] Orbit depth ablation (K=6, 12, 24) …")
    acc_depth = orbit_depth_ablation(
        clf, vae, generators, SEMs, imgs_0, lbls_0, ANGLES, dig2cls,
        eps=args.eps, K_list=[6, 12, 24])
 
    # ── Summary ───────────────────────────────────────────────────────────────
    mean_base    = np.mean(list(acc_baseline.values()))
    mean_uniform = np.mean(list(acc_uniform.values()))
    mean_swop    = np.mean(list(acc_swop.values()))
 
    print(f"\n{'='*60}")
    print(f"  RESULTS SUMMARY — Exp 2")
    print(f"{'='*60}")
    print(f"  MLP-NoAug (baseline)    mean acc = {mean_base:.4f}")
    print(f"  Uniform orbit avg       mean acc = {mean_uniform:.4f}  Δ = {mean_uniform-mean_base:+.4f}")
    print(f"  SWOP (ours)             mean acc = {mean_swop:.4f}  Δ = {mean_swop-mean_base:+.4f}")
    print(f"\n  Orbit depth results:")
    for K, acc in acc_depth.items():
        m = np.mean(list(acc.values()))
        print(f"    K={K:2d}  mean acc = {m:.4f}")
    print(f"\n  SWOP vs Uniform improvement = {mean_swop - mean_uniform:+.4f}")
    print(f"  (positive = SEM weighting helps)")
 
    # ── Save artefacts ────────────────────────────────────────────────────────
    print("\n[6] Saving results and figures …")
    np.savez(os.path.join(args.out_dir, "orbit_results.npz"),
             angles    = ANGLES,
             acc_baseline  = [acc_baseline[a]  for a in ANGLES],
             acc_uniform   = [acc_uniform[a]   for a in ANGLES],
             acc_swop      = [acc_swop[a]      for a in ANGLES],
             SEMs      = SEMs,
             gcs_scores = gcs_scores,
             K_ablation_k   = list(acc_depth.keys()),
             K_ablation_acc = [[acc_depth[K][a] for a in ANGLES] for K in acc_depth])
 
    plot_results(ANGLES, acc_baseline, acc_uniform, acc_swop, acc_depth,
                 gcs_scores, SEMs,
                 os.path.join(args.out_dir, "fig_orbit_comparison.png"))
 
    plot_orbit_images(vae, imgs_0, generators, SEMs, args.eps, K=8,
                      n_samples=3,
                      out_path=os.path.join(args.out_dir, "fig_orbit_images.png"))
 
    print(f"\n✅  Exp 2 complete.  Outputs → {args.out_dir}/")
    print(f"    SWOP mean accuracy = {mean_swop:.4f}  "
          f"(+{mean_swop - mean_base:.4f} over no-invariance baseline)")

In [13]:
if __name__ == "__main__":
    main()


  Bonus Task · Exp 2: SEM-Weighted Orbit Averaging (SWOP)
  device = cuda
  output → ./bonus_outputs/exp2

  ✓ VAE loaded (epoch 47)
  ✓ Task-3 generators loaded: 3 generators
      G_1  SEM = 0.9899
      G_2  SEM = 0.9869
      G_3  SEM = 0.9809
  ✓ Classifier ('noaug') loaded from /kaggle/input/models/vitthal2945/classifier-exp1/pytorch/default/1/classifiers_exp1.pt
  loading rotated_mnist_test.h5 ... done (0.6s)
  Test (0° only): 2167 samples  |  angles: [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]

[1] Evaluating baseline (MLP-NoAug, no orbit) …


  eval baseline:   0%|          | 0/12 [00:00<?, ?it/s]


[2] Evaluating uniform orbit averaging (all generators, equal weight) …


  eval uniform:   0%|          | 0/12 [00:00<?, ?it/s]


[3] Evaluating SWOP (SEM-weighted orbit pooling) …


...


    Orbit depth K=12 …


  eval SWOP K=12:   0%|          | 0/12 [00:00<?, ?it/s]


    Orbit depth K=24 …


  eval SWOP K=24:   0%|          | 0/12 [00:00<?, ?it/s]


  RESULTS SUMMARY — Exp 2
  MLP-NoAug (baseline)    mean acc = 0.8496
  Uniform orbit avg       mean acc = 0.8078  Δ = -0.0417
  SWOP (ours)             mean acc = 0.8079  Δ = -0.0416

  Orbit depth results:
    K= 6  mean acc = 0.8300
    K=12  mean acc = 0.8079
    K=24  mean acc = 0.7694

  SWOP vs Uniform improvement = +0.0001
  (positive = SEM weighting helps)

[6] Saving results and figures …
  saved → ./bonus_outputs/exp2/fig_orbit_comparison.png
  saved → ./bonus_outputs/exp2/fig_orbit_images.png

✅  Exp 2 complete.  Outputs → ./bonus_outputs/exp2/
    SWOP mean accuracy = 0.8079  (+-0.0416 over no-invariance baseline)
